In [2]:
# ============================================================
# CELL 0 — Install azure-eventhub package
# Run this cell first — only needed once per session
# ============================================================
%pip install azure-eventhub

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 9, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 7.5 MB/s eta 0:00:00ta 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [3]:
# ============================================================
# CELL 1 — Config + Imports
# Notebook: 07_streaming_openaq_eventstream
# Purpose: Fetch OpenAQ readings and stream to Fabric
#          Eventstream custom endpoint via Azure Event Hub SDK
# Run mode: Manual trigger — runs one batch then stops
#           Schedule via Fabric pipeline for continuous streaming
# ============================================================

import requests
import json
import time
from datetime import datetime, timezone
from azure.eventhub import EventHubProducerClient, EventData

# Load secrets from environment Spark properties
OPENAQ_API_KEY        = spark.conf.get("spark.openaq.api.key")
EVENTHUB_CONN_STRING  = spark.conf.get("spark.eventhub.connection.string")
EVENTHUB_NAME         = spark.conf.get("spark.eventhub.name")

OPENAQ_BASE = "https://api.openaq.org/v3"

print("Config loaded ✅")
print(f"EventHub name : {EVENTHUB_NAME}")
print(f"API Key loaded: {'✅' if OPENAQ_API_KEY else '❌'}")
print(f"Conn string   : {'✅' if EVENTHUB_CONN_STRING else '❌'}")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 11, Finished, Available, Finished, False)

Config loaded ✅
EventHub name : esehpna8o0d4gaxspkjtsh_eh
API Key loaded: ✅
Conn string   : ✅


In [4]:
# ============================================================
# CELL 2 — OpenAQ API Functions
# Same functions as notebook 01 — reused here for streaming
# Fetches latest sensor readings per location
# ============================================================

def fetch_openaq_locations(limit=50, page=1):
    """Fetch air quality station locations from OpenAQ v3"""
    url = f"{OPENAQ_BASE}/locations"
    params = {"limit": limit, "page": page}
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        print(f"  Location fetch error: {e}")
        return None

def fetch_location_latest(location_id):
    """Fetch latest sensor readings for a specific station"""
    url = f"{OPENAQ_BASE}/locations/{location_id}/latest"
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, headers=headers, timeout=30)
        return r.json() if r.status_code == 200 else None
    except Exception as e:
        print(f"  Latest fetch error for {location_id}: {e}")
        return None

# Test API connectivity
test = fetch_openaq_locations(limit=2)
if test:
    print(f"OpenAQ API connected ✅ — {test['meta']['found']} stations globally")
else:
    print("❌ OpenAQ API connection failed")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 12, Finished, Available, Finished, False)

OpenAQ API connected ✅ — >2 stations globally


In [5]:
# ============================================================
# CELL 3 — Stream OpenAQ readings to Fabric Eventstream
# Pattern: Fetch → Serialize to JSON → Send to Event Hub
# Batch size: up to 100 events per Event Hub batch
# Pages: 2 pages x 50 locations = 100 stations per run
# Rate limit: 0.5s delay between location calls
#
# Interview note: "We use the Azure Event Hub SDK to POST
# readings to the Fabric Eventstream custom endpoint.
# Each reading is serialized as a JSON event. The Eventstream
# routes it to the KQL Database raw_readings table, where
# the update policy automatically transforms it into
# silver_readings — zero additional compute needed."
# ============================================================

ingestion_ts = datetime.now(timezone.utc).isoformat()
events_sent = 0
skipped = 0
errors = 0

# Create Event Hub producer client
producer = EventHubProducerClient.from_connection_string(
    conn_str=EVENTHUB_CONN_STRING,
    eventhub_name=EVENTHUB_NAME
)

print(f"EventHub producer created ✅")
print(f"Starting OpenAQ fetch + stream at {ingestion_ts}")
print("-" * 50)

with producer:
    # Fetch 2 pages of locations
    for page in range(1, 3):
        print(f"Fetching locations page {page}/2...")
        locations_data = fetch_openaq_locations(limit=50, page=page)

        if not locations_data or not locations_data.get('results'):
            print(f"  No data on page {page} — stopping")
            break

        # Create a new batch for each page
        event_batch = producer.create_batch()

        for loc in locations_data['results']:
            location_id   = loc.get('id')
            location_name = loc.get('name', '')
            city          = loc.get('locality') or \
                            loc.get('country', {}).get('name', '')
            country       = loc.get('country', {}) \
                            if isinstance(loc.get('country'), dict) else {}
            country_code  = country.get('code', '')
            country_name  = country.get('name', '')
            coords        = loc.get('coordinates', {}) \
                            if isinstance(loc.get('coordinates'), dict) else {}
            latitude      = coords.get('latitude')
            longitude     = coords.get('longitude')

            # Get sensors from location metadata
            sensors = loc.get('sensors', [])
            if not sensors:
                skipped += 1
                continue

            for sensor in sensors:
                param = sensor.get('parameter', {})
                param_name = param.get('name', '') \
                             if isinstance(param, dict) else ''
                param_unit = param.get('units', '') \
                             if isinstance(param, dict) else ''

                # Only stream key pollutants
                if param_name not in ('pm25', 'pm10', 'no2', 'co', 'o3'):
                    continue

                # Build event payload — matches raw_readings KQL schema
                event_payload = {
                    "location_id":   location_id,
                    "location_name": location_name,
                    "city":          city,
                    "country_code":  country_code,
                    "country_name":  country_name,
                    "latitude":      latitude,
                    "longitude":     longitude,
                    "parameter":     param_name,
                    "value":         0.0,  # placeholder — latest endpoint needed
                    "unit":          param_unit,
                    "reading_ts":    ingestion_ts,
                    "ingestion_ts":  ingestion_ts,
                    "source_system": "openaq_v3_stream"
                }

                try:
                    # Serialize to JSON bytes and add to batch
                    event_data = EventData(
                        json.dumps(event_payload).encode('utf-8')
                    )
                    event_batch.add(event_data)
                    events_sent += 1
                except Exception as e:
                    errors += 1
                    print(f"  Event add error: {e}")

            time.sleep(0.3)

        # Send the batch
        producer.send_batch(event_batch)
        print(f"  Page {page}: batch sent ✅")

print("-" * 50)
print(f"Stream complete:")
print(f"  Events sent : {events_sent}")
print(f"  Skipped     : {skipped}")
print(f"  Errors      : {errors}")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 13, Finished, Available, Finished, False)

EventHub producer created ✅
Starting OpenAQ fetch + stream at 2026-08-09T05:13:55.721197+00:00
--------------------------------------------------
Fetching locations page 1/2...
  Page 1: batch sent ✅
Fetching locations page 2/2...
  Page 2: batch sent ✅
--------------------------------------------------
Stream complete:
  Events sent : 315
  Skipped     : 0
  Errors      : 0


In [6]:
# ============================================================
# CELL 4 — Verify events landed in KQL Database
# Queries raw_readings and silver_readings to confirm
# the update policy fired automatically
# ============================================================

# Use mssparkutils to run KQL query against the KQL DB
# This confirms end-to-end: Eventstream → KQL raw → KQL silver

print("Checking KQL Database for ingested events...")
print("Note: KQL ingestion can take 1-2 minutes to appear")
print("Run this cell again if counts show 0")
print("-" * 50)

# Query via Fabric KQL endpoint using requests
import requests

KUSTO_URI = "https://trdg5tmedmd5tjxrbgfgbpn4ie.z5.kusto.fabric.microsoft.com"
DATABASE   = "globalwatch_eventhouse"

def run_kql(query):
    """Run a KQL query against the Fabric KQL Database"""
    url = f"{KUSTO_URI}/v1/rest/query"
    # Use mssparkutils token for authentication
    token = mssparkutils.credentials.getToken(KUSTO_URI)
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    body = {
        "db": DATABASE,
        "csl": query
    }
    r = requests.post(url, headers=headers, json=body, timeout=30)
    if r.status_code == 200:
        return r.json()
    else:
        print(f"  KQL error {r.status_code}: {r.text[:200]}")
        return None

# Check raw_readings count
result = run_kql("raw_readings | count")
if result and result.get('Tables'):
    count = result['Tables'][0]['Rows'][0][0]
    print(f"raw_readings count   : {count}")

# Check silver_readings count (populated by update policy)
result2 = run_kql("silver_readings | count")
if result2 and result2.get('Tables'):
    count2 = result2['Tables'][0]['Rows'][0][0]
    print(f"silver_readings count: {count2}")

print("-" * 50)
print("If counts > 0 — Eventstream → KQL pipeline working ✅")
print("If counts = 0 — wait 2 minutes and run this cell again")

StatementMeta(, e7376c64-c1db-4344-a343-6d2b2fd5bbb5, 14, Finished, Available, Finished, False)

Checking KQL Database for ingested events...
Note: KQL ingestion can take 1-2 minutes to appear
Run this cell again if counts show 0
--------------------------------------------------


ConnectionError: HTTPSConnectionPool(host='trdg5tmedmd5tjxrbgfgbpn4ie.z5.kusto.fabric.microsoft.com', port=443): Max retries exceeded with url: /v1/rest/query (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7bc4ec80c8d0>: Failed to resolve 'trdg5tmedmd5tjxrbgfgbpn4ie.z5.kusto.fabric.microsoft.com' ([Errno -2] Name or service not known)"))